# 04｜模型載入與異常預測示範

本 notebook 載入 `models/random_forest_anomaly.joblib`，建立 5 筆可重現的隨機合成資料，並預測每筆是否屬於異常（0＝正常、1＝異常）。

為使隨機資料仍符合資料集的欄位與大致範圍，Notebook 會從清理資料中隨機抽取 5 筆作為基準，再對厚度測點及製程實際值加入小幅隨機擾動。這些資料只用於示範模型推論，不是新的真實批次，也不能用來評估模型效能。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# 從目前目錄或上層目錄尋找已訓練模型與資料準備輸出。
model_name = "random_forest_anomaly.joblib"
model_path = next((p / "models" / model_name for p in [Path.cwd(), *Path.cwd().parents] if (p / "models" / model_name).exists()), None)
if model_path is None:
    raise FileNotFoundError("找不到 models/random_forest_anomaly.joblib，請先執行 03_random_forest_training.ipynb。")
project_dir = model_path.parent.parent
reference_path = project_dir / "data" / "processed" / "fccl_cleaned.csv"
if not reference_path.exists():
    raise FileNotFoundError("找不到 data/processed/fccl_cleaned.csv，請先執行 02_data_preparation.ipynb。")

# 載入模型套件；模型內含訓練時的缺值填補與類別編碼 Pipeline。
bundle = joblib.load(model_path)
model = bundle["model"]
feature_columns = bundle["feature_columns"]
target_column = bundle["target_column"]
print(f"模型檔：{model_path}")
print(f"預測目標：{target_column}（0＝正常，1＝異常）")
print(f"模型特徵數：{len(feature_columns)}")

## 1. 產生 5 筆隨機合成資料

以固定亂數種子抽取 5 個不同的基準批次，再對測點厚度和製程實際值加入小幅隨機擾動。厚度平均、標準差和全距會依擾動後的測點重新計算，避免摘要欄位與測點完全脫節。類別欄位沿用抽中基準批次的值。

In [ ]:
reference = pd.read_csv(reference_path)
missing_features = sorted(set(feature_columns) - set(reference.columns))
if missing_features:
    raise ValueError(f"參考資料缺少模型欄位：{missing_features}")

rng = np.random.default_rng(2026)
# 5 筆資料各自以不同來源批次為基準，讓示範樣本涵蓋資料中的不同情況。
base_rows = reference.sample(n=5, replace=False, random_state=2026).reset_index(drop=True)
synthetic = base_rows[feature_columns].copy()
point_columns = [f"Point_{i}_um" for i in range(1, 11) if f"Point_{i}_um" in feature_columns]

# 以每筆樣本的測點變異量作為擾動尺度，維持厚度數值在合理鄰近範圍。
if point_columns:
    point_sd = synthetic[point_columns].std(axis=1).fillna(0).to_numpy()
    noise = rng.normal(loc=0, scale=np.maximum(point_sd * 0.05, 0.002)[:, None], size=(len(synthetic), len(point_columns)))
    synthetic.loc[:, point_columns] = synthetic[point_columns].to_numpy() + noise
    synthetic["Thickness_Mean_um"] = synthetic[point_columns].mean(axis=1)
    synthetic["Thickness_Std_um"] = synthetic[point_columns].std(axis=1)
    synthetic["Thickness_Range_um"] = synthetic[point_columns].max(axis=1) - synthetic[point_columns].min(axis=1)

# 製程設定目標保持基準值，實際值加入約 1% 目標量級的擾動。
for actual_col in [c for c in feature_columns if c.endswith("_actual")]:
    base = actual_col.removesuffix("_actual")
    target_col = f"{base}_target"
    if target_col in synthetic.columns:
        scale = synthetic[target_col].abs().to_numpy() * 0.01
        synthetic[actual_col] = synthetic[actual_col].to_numpy() + rng.normal(0, np.maximum(scale, 0.001), size=len(synthetic))

# 依模型原始欄位順序排列，避免推論時誤用或缺少欄位。
synthetic = synthetic.loc[:, feature_columns]
print(f"合成資料筆數：{len(synthetic)}")
display(synthetic)

## 2. 執行異常預測

使用模型的 Pipeline 自動套用訓練時的前處理。輸出預測類別、異常機率與正常機率；機率反映模型分數，不等同於實際異常發生率的保證。

In [ ]:
predicted_class = model.predict(synthetic)
class_probabilities = model.predict_proba(synthetic)
classes = list(model.named_steps["classifier"].classes_)
anomaly_column = classes.index(1)
normal_column = classes.index(0)

results = pd.DataFrame({
    "示範編號": [f"合成樣本 {i}" for i in range(1, len(synthetic) + 1)],
    "預測類別": predicted_class,
    "預測結果": ["異常" if label == 1 else "正常" for label in predicted_class],
    "正常機率": class_probabilities[:, normal_column],
    "異常機率": class_probabilities[:, anomaly_column],
})
results[["正常機率", "異常機率"]] = results[["正常機率", "異常機率"]].round(4)
display(results)
print("提醒：以上輸入是隨機擾動產生的合成示範資料，預測結果不可視為真實批次的判定。")

## 3. 檢視特徵與預測結果

可將預測結果和同一列合成輸入並排檢查。特徵欄位遵循訓練模型使用的欄位清單；如果未來使用真實資料，請先確認欄位定義、單位和量測流程與訓練資料一致。

In [ ]:
# 合併呈現輸入特徵與預測輸出，便於檢視每一筆示範資料。
prediction_review = pd.concat([results, synthetic.reset_index(drop=True)], axis=1)
display(prediction_review)

## 注意事項

- 這個 Notebook 僅示範載入模型與批次推論，不會重新訓練模型。
- 模型是以合成 FCCL 資料訓練；實際使用前需以代表性的真實資料驗證。
- 若更換模型檔，請確認其特徵欄位與此 Notebook 載入的 `feature_columns` 相容。